# 01 · Authoring fundamentals

**Story so far:** the platform connection works. Now we lay the foundation of Review
Radar: a first pipeline that generates a batch of raw reviews and runs a light
normalization pass. It's small enough to read in one sitting and real enough to establish
every concept the later chapters build on.

**Learning goals**

1. Understand `TaskEnvironment`, the unit of image/resource/config grouping in v2
2. Write sync and async tasks; know when each is appropriate
3. Build container images declaratively
4. Request CPU/memory/GPU/disk/shm resources, with request/limit ranges
5. Compose multi-environment pipelines and override configuration at call time

If you're coming from Flyte/Union **v1**: there is no `@workflow` decorator in v2;
composition is plain Python inside tasks. The full mapping is in
[09-migration-v1-to-v2](./09-migration-v1-to-v2.ipynb).

In [1]:
import flyte

flyte.init_from_config()

## 1. TaskEnvironment anatomy

A `TaskEnvironment` groups tasks that share an image, resources, and execution settings.
One environment ↔ one pod spec, roughly. The important constructor arguments:

| Argument | What it controls |
|---|---|
| `name` | Prefix for pod names and UI grouping |
| `image` | Container image (built for you, next section) |
| `resources` | CPU / memory / GPU / disk / shm per task pod |
| `env_vars` | Plain environment variables |
| `secrets` | Injected secrets ([02](./02-data-flow.ipynb)) |
| `depends_on` | Other environments this one references (deploy ordering + image reuse) |
| `cache` | Default cache policy for all tasks in the env ([04](./04-caching-and-reproducibility.ipynb)) |
| `reusable` | Warm-container pool ([05](./05-reusable-containers.ipynb)) |
| `interruptible` / `queue` | Spot instances and queue targeting ([04](./04-caching-and-reproducibility.ipynb)) |
| `plugin_config` | Hands the pod to a compute plugin, e.g. Ray ([06](./06-training-at-scale.ipynb)) |

`retries` and `timeout` are **per-task** (on `@env.task`), not per-environment.

In [2]:
env = flyte.TaskEnvironment(
    name="reviews",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    env_vars={"PIPELINE_STAGE": "workshop"},
)

## 2. The first Review Radar tasks: sync vs async

Both task styles work. Async is the **primary** model in v2 because it makes fan-out
natural (`asyncio.gather`, `flyte.map.aio`) and is required for concurrent execution
inside reusable containers. Rule of thumb:

- Task **calls other tasks** or does I/O-bound work → `async def`
- Leaf task doing pure CPU work → `def` is fine

`generate_reviews` synthesizes the raw dataset we'll use all series (in a customer
engagement, this is where your ingest goes: an API pull, a warehouse export, or a bucket
listing). `normalize` is a sync leaf task; `ingest_pipeline` orchestrates.

In [3]:
import random
from typing import List

PRODUCTS = ["espresso machine", "trail shoes", "noise-canceling headphones", "air fryer"]
PHRASES = {
    5: "absolutely love this {p}, exceeded expectations",
    4: "solid {p}, works as advertised",
    3: "the {p} is okay, nothing special",
    2: "disappointed with the {p}, quality issues",
    1: "terrible {p}, arrived broken and support ignored me",
}


@env.task
async def generate_reviews(n: int, seed: int = 7) -> List[dict]:
    """Synthesize raw reviews: the story's stand-in for a real ingest source."""
    rng = random.Random(seed)
    reviews = []
    for i in range(n):
        stars = rng.randint(1, 5)
        product = rng.choice(PRODUCTS)
        reviews.append({
            "id": i,
            "product": product,
            "stars": stars,
            "text": PHRASES[stars].format(p=product).upper() if rng.random() < 0.3
                    else PHRASES[stars].format(p=product),
        })
    return reviews


@env.task
def normalize(review: dict) -> dict:            # sync leaf task
    return {**review, "text": review["text"].strip().lower()}


@env.task
async def ingest_pipeline(n: int = 20) -> List[dict]:
    raw = await generate_reviews(n=n)
    cleaned = [await normalize.aio(r) for r in raw]   # call sync tasks from async with .aio
    return cleaned

### Local vs remote execution

- **Plain Python call**: `ingest_pipeline(5)` runs the function body in your kernel.
  No cluster, no tracking. Great for unit tests.
- **`flyte.run`**: ships the code to the cluster; every task call becomes a tracked,
  isolated container execution with its own resources and logs.

In [ ]:
run = flyte.run(ingest_pipeline, n=20)
print(run.url)
run.wait()
run.outputs()[:3]

> Sequentially normalizing one review at a time is deliberately naive; chapter
> [03](./03-processing-at-scale.ipynb) turns this into a proper parallel fan-out.

## 3. Container images

`flyte.Image` is a declarative, layered image builder: think `ImageSpec` (v1), rebuilt.
Start from `from_debian_base()` (ships with `flyte` pre-installed) and chain layers.
Images are **content-addressed**: identical definitions reuse the cached build, and they
land in whatever registry the deployment is configured with.

In [5]:
ml_image = (
    flyte.Image.from_debian_base(name="workshop-ml", python_version=(3, 12))
    .with_apt_packages("libgomp1")
    .with_pip_packages("scikit-learn==1.5.2", "pandas==2.2.3", "pyarrow")
    .with_env_vars({"OMP_NUM_THREADS": "2"})
)

# For images in a private registry that needs credentials, add a registry secret:
# private_image = flyte.Image.from_debian_base(
#     name="internal-tools",
#     registry="<your-registry>/<repo>",
#     registry_secret="registry-pull-secret",   # created with: flyte create secret ...
# )

# Only for non-Debian bases (e.g. CUDA): flyte is NOT pre-installed, add it yourself:
# cuda_image = (
#     flyte.Image.from_base("nvidia/cuda:12.4.1-runtime-ubuntu22.04")
#     .with_pip_packages("flyte==2.5.7", "torch")
# )

## 4. Resources

`flyte.Resources` covers CPU, memory, GPU, ephemeral disk, and shared memory. Two forms:

- Single value → request == limit: `cpu="2"`
- Tuple → (request, limit): `cpu=("1", "4")` (the pod can burst)

For GPUs, always name the device type: `gpu="1"` schedules on any GPU node, which on a
multi-pool cluster makes scheduling non-deterministic and can waste an A100 on a T4 job.
[Learn more about GPU support in Flyte](https://www.union.ai/docs/v2/union/user-guide/task-configuration/resources/#nvidia-gpus)

In [6]:
train_env = flyte.TaskEnvironment(
    name="training_preview",
    image=ml_image,
    resources=flyte.Resources(
        cpu=("2", "4"),                                # request 2, burst to 4
        memory=("4Gi", "8Gi"),
        disk="20Gi",                                   # ephemeral scratch space
        shm="2Gi",                                     # /dev/shm, e.g. torch DataLoader
        gpu=flyte.GPU(device="T4", quantity=1),        # or A100, L4, H100, ...
    ),
)

# GPU variants, depending on what the deployment exposes:
#   flyte.GPU(device="A100", quantity=2)
#   flyte.GPU(device="A100", quantity=1, partition="1g.5gb")   # MIG slice
#   flyte.TPU(device="V5P", partition="2x2x1")                 # TPUs (GCP)

## 5. Multi-environment pipelines

Real pipelines mix heavy and light steps. Put the heavy steps in their own environment and
keep the orchestrating task lightweight: you don't want the coordinator holding a GPU
while it waits. `depends_on` tells the deployment which environments travel together.
Review Radar will use this split from chapter [03](./03-processing-at-scale.ipynb)
onward: a light driver env orchestrating heavier worker envs.

In [7]:
light_env = flyte.TaskEnvironment(
    name="driver_preview",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    depends_on=[train_env],           # the driver calls tasks from train_env
)


@train_env.task
async def word_count(reviews: List[dict]) -> int:
    return sum(len(r["text"].split()) for r in reviews)


@light_env.task
async def stats_pipeline(n: int = 50) -> int:
    reviews = await generate_reviews(n=n)
    return await word_count(reviews=reviews)

## 6. `.override()`: call-time configuration

Any task call can override resources, retries, timeout, cache, queue... without touching
the task definition. This is how you get "same code, different sizing", and how you build
the error-recovery patterns you'll see in [03](./03-processing-at-scale.ipynb).

`word_count` is defined on `train_env`, so by default it pulls that env's heavy box,
including a T4 GPU. But counting words needs no GPU. Override this call down to plain
CPU, leaving the task definition (and its default for every other caller) untouched:

In [ ]:
@light_env.task
async def stats_pipeline_cpu(n: int = 50) -> int:
    reviews = await generate_reviews(n=n)
    # same word_count code, but THIS invocation drops the GPU and shrinks the box
    return await word_count.override(
        resources=flyte.Resources(cpu="1", memory="1Gi")
    )(reviews=reviews)


run = flyte.run(stats_pipeline_cpu, n=50)
print(run.url)
run.wait()
run.outputs()

> The task definition is the default; an override is the exception for one call, and it
> takes far more than resources (`retries`, `timeout`, `cache`, `queue`, `interruptible`,
> `env_vars`). Run plain `stats_pipeline` instead and `word_count` would request
> `train_env`'s T4 GPU; the override above is what lets a lightweight step skip it.

## Further reading

- Union docs: [Configure tasks](https://www.union.ai/docs/v2/union/user-guide/task-configuration/): images, resources, environments
- Next: [02-data-flow](./02-data-flow.ipynb), where the reviews stop being in-memory lists and
  become durable datasets in the object store